# 第 1 周练习 —— 公司宣传册生成器

## 练习目标（理念）

做一个小产品：输入**公司名称 + 官网**，自动抓取相关页面，再让大模型生成一份短宣传册，面向潜在客户、投资人与求职者。

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | `WebsiteScraper` 取链接与正文 |
| system / user prompts | 选链接、写宣传册两套提示词 |
| Structured JSON | `response_format={"type": "json_object"}` |
| One-shot prompting | system 里给一段 JSON 示例 |
| 流式输出 | 末尾 `stream=True` + `update_display` |

## 怎么跑

1. 确保同目录有可用的 `scraper.py`（提供 `WebsiteScraper`）
2. `.env` 里配置 `OPENAI_API_KEY`，自上而下运行单元格
3. 可把示例 URL 换成你想分析的公司主页


In [5]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（例如 API Key）
import os
# 导入标准库 json：把模型返回的 JSON 字符串解析成 Python 字典
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：调用 Chat Completions API
from openai import OpenAI
# 从本地 scraper 模块导入 WebsiteScraper：抓取页面链接与正文
from scraper import WebsiteScraper


In [ ]:
# ========== 环境 + 模型常量：鉴权体检与客户端 ==========

# 加载 .env；override=True 用文件值覆盖已有环境变量
load_dotenv(override=True)
# 读取 OpenAI API Key
api_key = os.getenv('OPENAI_API_KEY')
# 简单校验：存在、前缀 sk-proj-、长度大于 10（注意 >10 两边空格保持原样）
if api_key and api_key.startswith('sk-proj-') and len(api_key) >10:
    print("API key is valid")
else:
    print("API key is invalid")
# 后面选链接步骤默认用的模型名（字符串勿改）
MODEL = "gpt-5-nano"
# 创建 OpenAI 客户端（密钥来自环境变量）
openai = OpenAI()


## 第一步：让模型判断哪些链接相关

调用云端模型阅读网页上的链接列表，并以**结构化 JSON** 回复。

它要做两件事：

1. 判断哪些链接适合写进公司宣传册（About / Company / Careers 等）
2. 把相对路径（如 `/about`）补成完整 `https://...` URL

这里用 **one-shot prompting**：在 system prompt 里给一段「期望输出长什么样」的 JSON 示例，让模型照格式答。


In [8]:
# ========== system prompt：教模型如何筛链接并输出 JSON ==========
# 以下三引号内容是发给模型的指令，必须保持英文原样

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be the most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links":[
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [9]:
# ========== 组装 user prompt：把抓到的链接列表塞进提示 ==========

def get_links_user_prompt(url):
    """根据站点 URL 抓链接，拼成「请筛相关链接」的 user 提示词。"""
    # 先写任务说明（英文 prompt 保持原样）；{url} 会填入真实地址
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of the links are most relevant to include in a brochure about the company,
    respond with full https URL in JSON format.
    Do not include Terms of Service, privacy policy, email links.

    Links (some might be relative links):

    """
    # 用爬虫打开该站并取出所有链接
    scraper = WebsiteScraper(url)
    links = scraper.get_links()
    # 把链接列表用换行拼进 prompt 末尾
    user_prompt += "\n".join(links)
    return user_prompt


In [ ]:
# ========== 试跑：打印某站的「筛链接」user prompt，便于肉眼检查 ==========
print(get_links_user_prompt("https://www.flightaware.com/live/"))


In [11]:
# ========== 调用模型：选出与宣传册相关的链接（强制 JSON） ==========

def select_relevant_links(url):
    """请求 Chat Completions，解析 JSON，返回含 links 列表的字典。"""
    # 进度日志：当前 URL 与所用模型名
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    # 发起补全：system 定规则，user 带链接列表；response_format 强制 JSON 对象
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)  }
        ],
        response_format = {"type": "json_object"}
    )
    # 取出助手返回的 JSON 文本
    result = response.choices[0].message.content
    # 反序列化为 Python 字典（期望形如 {"links": [...]}）
    links = json.loads(result)
    # 打印筛出的条数
    print(f"Found {len(links['links'])} relevant links")
    return links


In [ ]:
# ========== 试跑：对示例站点做一次「相关链接」筛选 ==========
select_relevant_links("https://www.flightaware.com/live/")


## 第二步：制作宣传册

把落地页正文 + 相关子页内容组装成**另一段 user prompt**，再交给模型生成短宣传册。


In [13]:
# ========== 小工具：抓取单个页面的正文文本 ==========

def get_page_content(url):
    """用 WebsiteScraper 取 url 对应页面的可读文本。"""
    return WebsiteScraper(url).get_content()


In [14]:
# ========== 汇总落地页 + 相关链接页，拼成给模型的大段上下文 ==========

def fetch_page_and_all_relevant_links(url):
    """抓主页内容、筛相关链接，再把各页正文拼进一个 Markdown 风格字符串。"""
    # 先抓落地页正文
    contents = get_page_content(url)
    # 再让模型挑出相关链接
    relevant_links = select_relevant_links(url)
    # 开头放落地页；后面追加 Relevant Links 小节标题
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links: \n"
    # 遍历每条相关链接：写类型小标题，再追加页面正文
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        # 注意：此处原逻辑对 get_page_content 传入的是主站 url（非 link['url']），保持不改
        result += get_page_content(url)
    return result


In [ ]:
# ========== 试跑：打印「落地页 + 相关页」拼装结果（可能较长） ==========
print(fetch_page_and_all_relevant_links("https://www.flightaware.com/live/"))


In [16]:
# ========== 宣传册 prompts：system 定语气，user 塞公司素材 ==========

# 默认 system：正经、面向客户/投资人/招聘的短宣传册（英文指令勿改）
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# 若想换成幽默语气，可取消下面整段注释（演示用 prompt 控制 tone；英文正文保持原样）
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

def get_brochure_user_prompt(company_name, url):
    """拼装「写宣传册」的 user 提示：公司名 + 抓到的页面素材，并截断长度。"""
    # 英文任务说明；{company_name} 填公司名
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    # 追加抓取并汇总后的页面文本
    user_prompt += fetch_page_and_all_relevant_links(url)
    # 超过 5000 字符就截断，控制 token 用量与费用
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt


In [ ]:
# ========== 试跑：只生成 user prompt，不调用「写册」模型 ==========
get_brochure_user_prompt("FlightAware", "https://www.flightaware.com/live/")


In [18]:
# ========== 非流式生成宣传册，并用 Markdown 展示 ==========

# 导入 Markdown 展示类型（display 沿用前面单元格已导入的符号）
from IPython.display import Markdown


def create_brochure(company_name, url):
    """一次性拿到完整宣传册文本，再在笔记本里渲染为 Markdown。"""
    # 注意：这里模型写死为 gpt-4.1-mini（与前面 MODEL 常量可以不同）
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    # 取出助手正文
    result = response.choices[0].message.content
    # 在 Jupyter 里漂亮显示
    display(Markdown(result))


In [ ]:
# ========== 试跑：为 FlightAware 生成一版宣传册 ==========
create_brochure("FlightAware", "https://www.flightaware.com/live/")


## 小改进：德语 + 流式打字机效果

只需微调调用方式：打开 `stream=True`，一边收增量一边 `update_display`，
并换一套要求「全文用德语写宣传册」的 system prompt，就能看到熟悉的打字机动画。


In [20]:
# ========== 流式宣传册：德语输出 + 笔记本内增量刷新 ==========

# 导入 update_display：用同一个 display_id 反复更新同一块输出区
from IPython.display import update_display

# 德语版 system prompt（发给模型的英文指令保持原样）
brochure_system_prompt_german = """
You are a marketing assistant.

Generate the brochure entirely in German.
Use natural, professional German.
Keep headings, bullet points, and formatting in German.
"""

def stream_brochure(company_name, url):
    """流式调用：边生成边刷新 Markdown，模拟打字机效果。"""
    # stream=True：服务端持续推送增量 delta，而不是等整段完成
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt_german},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    # response：累积已收到的全部文本
    response = ""
    # 先占位显示空 Markdown，拿到可更新的 display_id
    display_handle = display(Markdown(""), display_id=True)
    # 逐块消费流式事件
    for chunk in stream:
        # delta.content 可能为 None，用 or '' 避免拼接报错
        response += chunk.choices[0].delta.content or ''
        # 用同一 display_id 刷新，形成打字机动画
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 试跑：流式生成德语宣传册 ==========
stream_brochure("FlightAware", "https://www.flightaware.com/live/")
